In [2]:
import numpy as np 
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt 
import warnings 
warnings.filterwarnings("ignore")

In [5]:
df = pd.read_csv("lulc_training_dataset.csv")

In [6]:
print(f"shape:{df.shape}")
df.info()
print("isnull:",df.isnull().sum())
print("isduplicate:",df.duplicated().sum())

shape:(27000, 43)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27000 entries, 0 to 26999
Data columns (total 43 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   system:index  27000 non-null  int64  
 1   Q1_B11        27000 non-null  float64
 2   Q1_B12        27000 non-null  float64
 3   Q1_B2         27000 non-null  float64
 4   Q1_B3         27000 non-null  float64
 5   Q1_B4         27000 non-null  float64
 6   Q1_B5         27000 non-null  float64
 7   Q1_B6         27000 non-null  float64
 8   Q1_B7         27000 non-null  float64
 9   Q1_B8         27000 non-null  float64
 10  Q1_B8A        27000 non-null  float64
 11  Q2_B11        27000 non-null  float64
 12  Q2_B12        27000 non-null  float64
 13  Q2_B2         27000 non-null  float64
 14  Q2_B3         27000 non-null  float64
 15  Q2_B4         27000 non-null  float64
 16  Q2_B5         27000 non-null  float64
 17  Q2_B6         27000 non-null  float64
 18  Q2_B7   

In [17]:
X = df[['Q1_B11', 'Q1_B12', 'Q1_B2', 'Q1_B3', 'Q1_B4', 'Q1_B5',
       'Q1_B6', 'Q1_B7', 'Q1_B8', 'Q1_B8A', 'Q2_B11', 'Q2_B12', 'Q2_B2',
       'Q2_B3', 'Q2_B4', 'Q2_B5', 'Q2_B6', 'Q2_B7', 'Q2_B8', 'Q2_B8A',
       'Q3_B11', 'Q3_B12', 'Q3_B2', 'Q3_B3', 'Q3_B4', 'Q3_B5', 'Q3_B6',
       'Q3_B7', 'Q3_B8', 'Q3_B8A', 'Q4_B11', 'Q4_B12', 'Q4_B2', 'Q4_B3',
       'Q4_B4', 'Q4_B5', 'Q4_B6', 'Q4_B7', 'Q4_B8', 'Q4_B8A']]
y = df["label"]

label
10    3000
20    3000
30    3000
40    3000
50    3000
60    3000
80    3000
90    3000
95    3000
Name: count, dtype: int64

In [12]:
from sklearn.model_selection import train_test_split 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, shuffle=True, stratify=y
)

In [18]:
from sklearn.ensemble import RandomForestClassifier 

rf = RandomForestClassifier(
    n_estimators=100,
    criterion="gini",
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [20]:
y_pred = rf.predict(X_test)

In [21]:
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score
print(classification_report(y_pred,y_test))

              precision    recall  f1-score   support

          10       0.81      0.80      0.80       607
          20       0.79      0.82      0.80       573
          30       0.75      0.80      0.77       561
          40       0.91      0.91      0.91       605
          50       0.85      0.81      0.83       636
          60       0.88      0.86      0.87       617
          80       0.97      1.00      0.99       586
          90       0.86      0.79      0.82       651
          95       0.88      0.93      0.90       564

    accuracy                           0.85      5400
   macro avg       0.85      0.86      0.85      5400
weighted avg       0.86      0.85      0.85      5400



In [22]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_xgb = le.fit_transform(y_train)
y_test_xgb = le.transform(y_test)

le.classes_

array([10, 20, 30, 40, 50, 60, 80, 90, 95])

In [23]:
from xgboost import XGBClassifier 

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softmax",
    num_class=9,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train_xgb)

,objective,'multi:softmax'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,'mlogloss'


In [24]:
y_pred_xgb_encoded = xgb.predict(X_test)
y_pred_xgb = le.inverse_transform(y_pred_xgb_encoded)

In [25]:
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))

Accuracy: 0.8701851851851852

Classification Report:
              precision    recall  f1-score   support

          10       0.82      0.80      0.81       600
          20       0.83      0.81      0.82       600
          30       0.82      0.79      0.80       600
          40       0.92      0.92      0.92       600
          50       0.83      0.88      0.85       600
          60       0.87      0.89      0.88       600
          80       1.00      0.97      0.99       600
          90       0.82      0.87      0.84       600
          95       0.93      0.89      0.91       600

    accuracy                           0.87      5400
   macro avg       0.87      0.87      0.87      5400
weighted avg       0.87      0.87      0.87      5400

